In [1]:
from google.colab import files
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
import csv

In [2]:
uploaded_files = files.upload()

data=np.genfromtxt('Dedup_4vert2Max5MutationsAcyclicLabeled.csv',delimiter=',')

Saving Dedup_4vert2Max5MutationsAcyclicLabeled.csv to Dedup_4vert2Max5MutationsAcyclicLabeled.csv


In [3]:
SIZE = 295502
data = np.zeros((295502, 16 + 8))

def ternary_to_int(strT):
  if strT=="T": return 1.0
  elif strT=="F": return 0.0
  else: return 0.5

def parse_Alex(strA):
  if strA=="F": return -0.5
  else: return strA[1:-1].split(" ")[1]

row_parse = lambda r : ([int(x) for x in r["quiver exchange matrix"][1:-1].split(" ")] +
                        [ternary_to_int(r["mutation finite"])] + [ternary_to_int(r["surface quiver"])] +
                        [int(r["determinant of exchange matrix"])] +  [int(r["determinant of companion modulo 4"])] +
                        [int(r["rank of exchange matrix"])] +  [parse_Alex(r["Alexander polynomial"])] +
                        [int(r["minimum number of arrows in class"])] + [ternary_to_int(r["mutation acyclic"])])

In [4]:
with open('Dedup_4vert2Max5MutationsAcyclicLabeled.csv', 'r', newline='') as f:
  reader = csv.DictReader(f)
  header = reader.fieldnames

  for i, row in enumerate(reader):
    data[i,:] = row_parse(row)

In [5]:
data.shape

(295502, 24)

In [6]:
t = data[:,-1]
Quivers = data[:,:23]

In [7]:
X_train, X_test, t_train, t_test = train_test_split(Quivers, t, test_size=0.3)

In [8]:
num_classes = 2
input_shape = (23,)

X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

X_train shape: (206851, 23, 1)
206851 train samples
88651 test samples


In [9]:

# convert class vectors to binary class matrices
t_train = keras.utils.to_categorical(t_train, num_classes)
t_test = keras.utils.to_categorical(t_test, num_classes)

In [10]:
max_faces = 128
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Dense(max_faces, activation='sigmoid'),
        layers.Dense(max_faces, activation='sigmoid'),
        layers.Dense(max_faces, activation='sigmoid'),
        layers.Dense(num_classes, activation="sigmoid", name="all_active"),
    ]
)

In [11]:
model.get_layer(name="all_active").trainable=False
model.get_layer('all_active').set_weights([np.array([[1.0]*max_faces, [-1.0]*max_faces]).T, np.array([-max_faces +0.5, max_faces - 0.5])])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ all_active (Dense)              │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,354 (142.01 KB)

 Trainable params: 36,096 (141.00 KB)

 Non-trainable params: 258 (1.01 KB)

In [12]:
batch_size = 100
epochs = 20

optimizer = keras.optimizers.Adam(learning_rate=0.005)

model.compile(loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

model.fit(X_train, t_train, batch_size=batch_size, epochs=epochs)

Epoch 1/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8724 - loss: 0.4921
Epoch 2/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9925 - loss: 0.1971
Epoch 3/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9928 - loss: 0.1958
Epoch 4/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9941 - loss: 0.1935
Epoch 5/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9938 - loss: 0.1941
Epoch 6/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9944 - loss: 0.1931
Epoch 7/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9945 - loss: 0.1927
Epoch 8/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9949 - loss: 0.1921
Epoch 9/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9952 - loss: 0.1918
Epoch 10/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9950 - loss: 0.1918
Epoch 11/20
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9955 - loss: 0.1913
Epoch 12/20
2069/2069 ━━━━━━━━

In [13]:
score = model.evaluate(X_test, t_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.18836872279644012
Test accuracy: 0.9967625737190247
